In [1]:
import os
import json
import warnings
from itertools import islice
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve)
 
warnings.filterwarnings("ignore")
SEED = 42
B_BOOT = int(os.environ.get("B_BOOT", 500))
HORIZONS = [1, 2, 3]
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)
 
df = pd.read_csv("data/processed/features.csv")
groups = pd.read_csv("results/feature_groups.csv", index_col=0)["group"]
FEATURE_SETS = {"static": groups[groups != "dynamics"].index.tolist(),
                "full": groups.index.tolist()}
tr = df[df.split == "train"].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
te = df[df.split == "test"].reset_index(drop=True)
 
 
# ------------------------------------------------------------------ models
def fit_lr(Xtr, ytr, Xva, yva):
    best = None
    for C in (0.01, 0.1, 1.0):
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(C=C, class_weight="balanced",
                                             max_iter=3000))
        m.fit(Xtr, ytr)
        ap = average_precision_score(yva, m.predict_proba(Xva)[:, 1])
        if best is None or ap > best[0]:
            best = (ap, C, m)
    _, C, m = best
    return {"model": m, "predict": lambda X: m.predict_proba(X)[:, 1],
            "info": {"C": C}}
 
 
def fit_rf(Xtr, ytr, Xva, yva):
    m = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=500, min_samples_leaf=20,
                               max_features="sqrt",
                               class_weight="balanced_subsample",
                               n_jobs=-1, random_state=SEED))
    m.fit(Xtr, ytr)
    return {"model": m, "predict": lambda X: m.predict_proba(X)[:, 1],
            "info": {"n_estimators": 500, "min_samples_leaf": 20}}
 
 
def fit_hgb(Xtr, ytr, Xva, yva):
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=50, l2_regularization=1.0,
        class_weight="balanced", early_stopping=False, random_state=SEED)
    m.fit(Xtr, ytr)
    aps = [average_precision_score(yva, p[:, 1])
           for p in m.staged_predict_proba(Xva)]
    best = int(np.argmax(aps)) + 1
    return {"model": m, "info": {"best_iter": best},
            "predict": lambda X: next(islice(
                m.staged_predict_proba(X), best - 1, best))[:, 1]}
 
 
def fit_xgb(Xtr, ytr, Xva, yva):
    from xgboost import XGBClassifier
    spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
    m = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8,
                      min_child_weight=5, reg_lambda=2.0,
                      scale_pos_weight=spw, eval_metric="aucpr",
                      early_stopping_rounds=50, tree_method="hist",
                      n_jobs=-1, random_state=SEED)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return {"model": m, "predict": lambda X: m.predict_proba(X)[:, 1],
            "info": {"best_iteration": int(m.best_iteration)}}
 
 
def fit_ebm(Xtr, ytr, Xva, yva):
    from interpret.glassbox import ExplainableBoostingClassifier
    w = np.where(ytr == 1, (ytr == 0).sum() / max((ytr == 1).sum(), 1), 1.0)
    m = ExplainableBoostingClassifier(random_state=SEED)
    m.fit(Xtr, ytr, sample_weight=w)
    return {"model": m, "predict": lambda X: m.predict_proba(X)[:, 1],
            "info": {}}
 
 
MODELS = {"logreg": fit_lr, "rf": fit_rf}
try:
    import xgboost  # noqa: F401
    MODELS["xgb"] = fit_xgb
except Exception:
    print("xgboost not installed -> using sklearn HistGradientBoosting "
          "as 'hgb' (install xgboost before the SHAP step)")
    MODELS["hgb"] = fit_hgb
try:
    import interpret  # noqa: F401
    MODELS["ebm"] = fit_ebm
except Exception:
    print("interpret not installed -> EBM skipped")
 
 
# ----------------------------------------------------------------- metrics
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))
 
 
def platt(s_val, y_val, others, is_prob):
    f = logit if is_prob else (lambda s: np.asarray(s, float))
    lr = LogisticRegression(C=1e6, max_iter=1000)
    lr.fit(f(s_val).reshape(-1, 1), y_val)
    return [lr.predict_proba(f(s).reshape(-1, 1))[:, 1]
            for s in [s_val] + others]
 
 
def f2_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f2 = 5 * prec[:-1] * rec[:-1] / np.maximum(4 * prec[:-1] + rec[:-1],
                                               1e-12)
    return float(thr[int(np.argmax(f2))])
 
 
def metrics(y, s, p, thr):
    y = np.asarray(y)
    prec, rec, _ = precision_recall_curve(y, s)
    ap = float(-np.sum(np.diff(rec) * prec[:-1]))
    pred = p >= thr
    tp = float((pred & (y == 1)).sum())
    fp = float((pred & (y == 0)).sum())
    fn = float((~pred & (y == 1)).sum())
    f2 = 5 * tp / max(5 * tp + 4 * fn + fp, 1e-12)
    brier = float(np.mean((p - y) ** 2))
    prev = float(y.mean())
    return {"ROC_AUC": float(roc_auc_score(y, s)), "PR_AUC": ap,
            "PR_lift": ap / prev,
            "recall@prec5%": float(rec[prec >= 0.05].max()),
            "recall@prec10%": float(rec[prec >= 0.10].max()),
            "F2": f2, "Brier": brier,
            "Brier_skill": 1 - brier / (prev * (1 - prev))}
 
 
# --------------------------------------------- bootstrap over test companies
comp = te.groupby("company_name").indices
idx_lists = list(comp.values())
rng = np.random.default_rng(SEED)
boots = [np.concatenate([idx_lists[i] for i in
                         rng.integers(0, len(idx_lists), len(idx_lists))])
         for _ in range(B_BOOT)]
 
rows, preds, boot_store, info_all = [], [], {}, {}
 
 
def evaluate(name, fs, h, s_va, s_te, is_prob):
    yv, yt = va[f"y{h}"].values, te[f"y{h}"].values
    p_va, p_te = platt(s_va, yv, [s_te], is_prob)
    thr = f2_threshold(yv, p_va)
    for split, y, s, p in (("val", yv, s_va, p_va), ("test", yt, s_te, p_te)):
        r = {"model": name, "features": fs, "horizon": h, "split": split,
             "n": len(y), "positives": int(y.sum()), "F2_threshold": thr}
        r.update(metrics(y, s, p, thr))
        if split == "test":
            bs = [metrics(yt[i], s[i], p[i], thr) for i in boots
                  if yt[i].sum() > 0]
            bdf = pd.DataFrame(bs)
            boot_store[(name, fs, h)] = bdf
            for c in bdf.columns:
                r[f"{c}_lo"], r[f"{c}_hi"] = np.percentile(bdf[c], [2.5, 97.5])
        rows.append(r)
    for split, d, s, p in (("val", va, s_va, p_va), ("test", te, s_te, p_te)):
        preds.append(pd.DataFrame({
            "company_name": d.company_name.values, "year": d.year.values,
            "split": split, "horizon": h, "model": name, "features": fs,
            "y": d[f"y{h}"].values, "score": s, "prob": p}))
 
 
for h in HORIZONS:
    print(f"\n##### horizon {h}")
    # Altman baseline (score = -Z, unfitted; only Platt on val)
    evaluate("altman_z", "-", h, -va.altman_z.values, -te.altman_z.values,
             is_prob=False)
    for fs, cols in FEATURE_SETS.items():
        Xtr, Xva, Xte = tr[cols], va[cols], te[cols]
        ytr, yva = tr[f"y{h}"].values, va[f"y{h}"].values
        for name, fit in MODELS.items():
            try:
                b = fit(Xtr, ytr, Xva, yva)
            except Exception as e:  # keep the run going
                print(f"  SKIPPED {name}/{fs}/h{h}: {e}")
                continue
            evaluate(name, fs, h, b["predict"](Xva), b["predict"](Xte),
                     is_prob=True)
            info_all[f"{name}|{fs}|h{h}"] = b["info"]
            joblib.dump({"model": b["model"], "cols": cols,
                         "info": b["info"]},
                        f"models/{name}_{fs}_h{h}.joblib")
            last = [r for r in rows if r["split"] == "test"][-1]
            print(f"  {name:7s} {fs:6s} test PR-AUC {last['PR_AUC']:.3f} "
                  f"ROC {last['ROC_AUC']:.3f}")
 
res = pd.DataFrame(rows)
res.to_csv("results/step4_metrics.csv", index=False)
pd.concat(preds).to_csv("data/processed/preds.csv", index=False)
with open("results/step4_model_info.json", "w") as f:
    json.dump(info_all, f, indent=1)
 
# paired bootstrap vs Altman (same company resamples)
pr_rows = []
for (name, fs, h), bdf in boot_store.items():
    if name == "altman_z":
        continue
    base = boot_store[("altman_z", "-", h)]
    n = min(len(bdf), len(base))
    for m in ("PR_AUC", "ROC_AUC"):
        d = bdf[m].values[:n] - base[m].values[:n]
        pr_rows.append({"model": name, "features": fs, "horizon": h,
                        "metric": m, "delta_mean": d.mean(),
                        "ci_lo": np.percentile(d, 2.5),
                        "ci_hi": np.percentile(d, 97.5),
                        "share_delta_le_0": float((d <= 0).mean())})
pd.DataFrame(pr_rows).to_csv("results/step4_paired_vs_altman.csv",
                             index=False)
 
show = res[(res.split == "test") & (res.features.isin(["full", "-"]))]
cols = ["model", "horizon", "positives", "ROC_AUC", "PR_AUC", "PR_lift",
        "recall@prec5%", "F2", "Brier_skill"]
print("\n=== TEST, full feature set (point estimates) ===")
print(show[cols].round(3).to_string(index=False))
 


##### horizon 1
  logreg  static test PR-AUC 0.092 ROC 0.896
  rf      static test PR-AUC 0.199 ROC 0.917
  xgb     static test PR-AUC 0.199 ROC 0.919
  ebm     static test PR-AUC 0.201 ROC 0.921
  logreg  full   test PR-AUC 0.096 ROC 0.882
  rf      full   test PR-AUC 0.195 ROC 0.922
  xgb     full   test PR-AUC 0.204 ROC 0.910
  ebm     full   test PR-AUC 0.189 ROC 0.912

##### horizon 2
  logreg  static test PR-AUC 0.093 ROC 0.867
  rf      static test PR-AUC 0.222 ROC 0.896
  xgb     static test PR-AUC 0.242 ROC 0.892
  ebm     static test PR-AUC 0.174 ROC 0.889
  logreg  full   test PR-AUC 0.094 ROC 0.857
  rf      full   test PR-AUC 0.214 ROC 0.894
  xgb     full   test PR-AUC 0.202 ROC 0.887
  ebm     full   test PR-AUC 0.172 ROC 0.888

##### horizon 3
  logreg  static test PR-AUC 0.096 ROC 0.837
  rf      static test PR-AUC 0.220 ROC 0.868
  xgb     static test PR-AUC 0.221 ROC 0.870
  ebm     static test PR-AUC 0.139 ROC 0.851
  logreg  full   test PR-AUC 0.093 ROC 0.826
  rf